# Cross-Domain Survival Evaluation — Full Pipeline

This notebook orchestrates the project by calling each root-level `.py` script in order.  
Markdown cells document purpose, inputs, and outputs; code cells only run scripts.

**Scientific goals (in order):**
1. **Reproduce** published numbers from three non-clinical survival baselines.
2. **Re-evaluate** the *same* frozen models under the anchor paper’s assumption-aligned ladder (*Stop Chasing the C-index*, ICML 2026).

| Domain | Baseline | Reproduction target |
|--------|----------|---------------------|
| 1 Engineering | Ahmed & Green 2024 (Backblaze Cox) | C-index **0.958** |
| 2 Credit | Bone-Winkel & Reichenbach 2024 (Bondora) | Cox + XGB-Cox ratings vs platform |
| 3 Platforms | Abedi Firouzjaei 2022 (Stack Exchange RSF) | C-index **0.66–0.76** |

Clone-and-run: **`REPRODUCE.md`**. Envs: `./bootstrap_envs.sh` (main `.venv`) and optionally `./bootstrap_envs.sh --d3` (PySurvival). Domain-3 details: `domain3-abedi-2022/CODE_ACCESS.md`.

---
## How to use
- Prefer selecting the **`.venv`** kernel after Block A creates it (A02). A02b creates Domain-3 if `conda` is available.
- Run from the **project root** (directory with `src/`, `A00_config.py`, `requirements.lock`).
- Execute cells top-to-bottom. Hard gates: finish **D** (reproduction) before **E** (anchor ladder).
- **Kaggle credentials required** before Block B (`B01` Bondora) — see REPRODUCE.md.
- Prefer fail-fast for a full reproduction: `export CDS_STRICT=1` (default still `WAITING` → exit 0).
- Two interpreters by design: **main `.venv`** (A02) for almost everything; **`d3-pysurvival`** (A02b, via conda) only for `D02_DOMAIN_03_train_rsf.py`.
- After D02, **D02b** freezes sksurv RSF joblibs (main `.venv`) required by Block E.


In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

from src.repro import (  # noqa: E402
    D3_PYSURVIVAL_SCRIPTS,
    resolve_d3_python,
    resolve_main_python,
    resolve_script_python,
)

_main = resolve_main_python(ROOT)
_d3 = resolve_d3_python()
print(f"Main trunk interpreter: {_main}")
print(f"Domain-3 interpreter:   {_d3 or '(not found — set CDS_D3_PYTHON or ./bootstrap_envs.sh --d3)'}")
if Path(_main).resolve() != Path(sys.executable).resolve():
    print(
        f"NOTE: notebook kernel is {sys.executable}\n"
        f"      Pipeline scripts will use {_main} (prefer selecting .venv as the kernel)."
    )


def run(script: str, extra_args: list[str] | None = None) -> None:
    """Execute a pipeline script with the env required for that stage."""
    path = ROOT / script
    if not path.exists():
        raise FileNotFoundError(
            f"{script} not found under {ROOT}. "
            "Implement this stage before running the cell."
        )
    exe = resolve_script_python(script, root=ROOT)
    if Path(script).name in D3_PYSURVIVAL_SCRIPTS:
        if resolve_d3_python():
            print(f"(D3 interpreter: {exe})", flush=True)
        else:
            print(
                "WARNING: CDS_D3_PYTHON / d3-pysurvival not found; "
                "falling back to main trunk (D02 may fail without PySurvival).",
                flush=True,
            )
    cmd = [exe, "-W", "default", str(path)]
    if extra_args:
        cmd.extend(extra_args)
    print(f"\n>>> {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, cwd=str(ROOT))
    if proc.returncode != 0:
        raise RuntimeError(f"{script} exited with code {proc.returncode}")
    print(f"<<< {script} OK", flush=True)


---
## Block A — Foundation

Validate config, create directories, install dependencies, then verify imports.  
These steps should always pass before any data download or modelling.

### A00 — Configuration validation

Loads `src/config.py` and checks internal consistency (domains, seeds, eval placeholders, literature paths).

**Output:** printable summary only — no files written.

In [ ]:
run("A00_config.py")

### A01 — Directory setup

Creates the live write targets from `cfg.DIRS`: `data/{raw,interim,processed}`, `results/{models,ladder,probes,logs,reproduction,harness,paper/...}`. Idempotent; adds `.gitkeep` only to otherwise empty dirs.

**Not created** (retired): `literature/`, `notebooks/`, `tests/`, `results/{metrics,figures}`, `data/interim/domain3`, `temp/`. Baseline PDFs live under `domain*-*/`; manuscript figures under `results/paper/figures/`.

**Output:** directory tree on disk.


In [ ]:
run("A01_setup_dirs.py")

### A02 — Install dependencies (main `.venv`)

Creates ``.venv`` if missing and installs ``requirements.lock`` **into that venv** (not into the system Python). Domain-3 PySurvival is **A02b** (next cell).


In [ ]:
run("A02_install_deps.py")

### A02b — Domain-3 env (`d3-pysurvival`)

Creates the **separate** conda env used only by `D02_DOMAIN_03_train_rsf.py` (PySurvival 0.1.2). Requires `conda` on `PATH`. Idempotent if the env already imports `pysurvival`. If the build hits the macOS `tp_print` ABI issue, see `domain3-abedi-2022/CODE_ACCESS.md` (or set `CDS_STRICT=1` to fail-fast).


In [ ]:
run("A02b_bootstrap_d3_env.py")
# Refresh resolvers after A02b writes CDS_D3_PYTHON / results/logs/d3_python_path.txt
from src.repro import resolve_d3_python, resolve_main_python
import os
hint = ROOT / "results" / "logs" / "d3_python_path.txt"
if hint.exists():
    os.environ["CDS_D3_PYTHON"] = hint.read_text().strip()
print("Main:", resolve_main_python(ROOT))
print("D3:  ", resolve_d3_python() or "(unavailable — D02 will warn)")


### A03 — Environment check + machine fingerprint

Imports every required package and prints versions for the audit trail.  
Also writes `results/logs/machine_fingerprint.json` (serial/UUID redacted).  
Standalone: `A04_machine_fingerprint.py`.

**Output:** pass/fail on stdout. Failures point back to A02.

In [ ]:
run("A03_env_check.py")
run("A04_machine_fingerprint.py")  # idempotent refresh of host fingerprint


---
## Block B — Data acquisition

Download public datasets and write an auditable manifest.  
**Order is fixed:** `B00 → B01 → B02 → B03` (no reordering by difficulty).

### B00 — Backblaze Drive Stats (Domain 1)

Downloads annual/quarterly zips for the Ahmed & Green window **2013–2022** into `data/raw/backblaze/`.  
Idempotent (skips complete files; resumes partial downloads).

In [ ]:
run("B00_download_backblaze.py")

### B01 — Bondora LoanData (Domain 2, canonical)

Acquires **`data/raw/bondora/LoanData.csv`** (Kaggle `marcobeyer/bondora-p2p-loans`) with pinned SHA-256.
Order: existing hash-ok → Kaggle API (`~/.kaggle/kaggle.json`).

Optional: portal `loan_dataset_investor.xlsx` (wrong schema; not used by DOMAIN_02 Cox).  
Writes `results/logs/b01_bondora_status.json` + `data/raw/bondora/SOURCE.md`.


In [ ]:
run("B01_download_bondora.py")

### B02 — Stack Exchange / Abedi artefacts (Domain 3)

Downloads the author’s GitHub release used in the paper:  
`processed_data/{p,ds,cs}/user_features.csv` (RSF inputs) plus matching `raw_data/` extracts.  
This is the preferred reproduction source (roadmap); Archive.org dumps are optional upstream.

In [ ]:
run("B02_download_stackexchange.py")

### B03 — Data manifest

Hashes every file under `data/raw/` (SHA-256), records sizes and source URLs, and writes:
- `results/logs/data_manifest.json`
- `results/logs/data_manifest.md`

Gate B→C: all expected Domain 1–3 artefacts must be present.

In [ ]:
run("B03_data_manifest.py")

---
## Block C — Protocol freeze (roadmap Phase D, early)

Freeze hypotheses H1–H5 + H_meta, C00.1–C00.3 decision rules, metrics, and reproduction targets **before** assumption-aligned evaluation (Block E).

Split policy is **not** frozen here — each Domain lane uses the paper's own protocol (`D01_DOMAIN_0n_*`).


### C00 — Preregister protocol freeze

Writes `results/logs/protocol_freeze.json` (+ `.md`) from `cfg.PROTOCOL`.  
Idempotent if unchanged; refuses overwrite unless `PROTOCOL["version"]` is bumped.


In [ ]:
run("C00_preregister_protocol.py")


---
## Block D — Faithful reproduction (roadmap Phase A)

Three **parallel lanes** after C00. Naming: `D{nn}_DOMAIN_0{d}_{slug}.py`.

| Lane | Baseline | Target | Status |
|------|----------|--------|--------|
| `DOMAIN_01` | Ahmed & Green 2024 | C-index **0.958** | **complete** (H6a; ours≈0.9595) |
| `DOMAIN_02` | Bone-Winkel & Reichenbach 2024 | Cox linear + boosted ratings | **complete** |
| `DOMAIN_03` | Abedi Firouzjaei 2022 | RSF C-index Table 8 / **0.66–0.76** | **complete** (1ª passagem) |

Merge with `D99_ALL_reproduction_report.py` before Block E.


### Lane DOMAIN_01 — Ahmed & Green (Backblaze / Cox)

Paper split/cohort live in `cfg.DOMAIN_01` and `D01_*`. Fase A lane **complete**.

**Cox GOF cohort (H6a, locked 2026-07-12):** healthy with calendar span >7y **∪** all failed → in-sample C≈**0.9595** (paper **0.958**, gap **+0.0015**).  
GitLab author URL is **404**. Eval/ladder uses `data/processed/domain1/drives_cox_cohort.parquet`.


#### D00 — Features / inventory

Builds `drives.parquet` + H6a flag (`cox_fit_h6a`) / `drives_cox_cohort.parquet`.  
If the zip stream already exists: `python -W default D00_DOMAIN_01_features_backblaze.py --amend-existing`.


In [ ]:
run("D00_DOMAIN_01_features_backblaze.py")


#### D01 — Split policy (Ahmed)


In [ ]:
run("D01_DOMAIN_01_split_ahmed.py")


#### D02 — Train Cox PH

Fits `lifelines.CoxPHFitter` on **H6a** only (`cfg.DOMAIN_01['cox_fit_population']`). Target C=0.958.


In [ ]:
run("D02_DOMAIN_01_train_cox.py")


#### D03 — Reproduction table (paper asset)

Writes mixed paper/ours/gap tables:
`results/reproduction/DOMAIN_01_reproduction_table.{json,csv,md,tex}`


In [ ]:
run("D03_DOMAIN_01_gap.py")


### Lane DOMAIN_02 — Bone-Winkel & Reichenbach (Bondora / Cox + ratings)

**Fase A complete.** Data: Kaggle [`marcobeyer/bondora-p2p-loans`](https://www.kaggle.com/api/v1/datasets/download/marcobeyer/bondora-p2p-loans) →
`D00` align (36m, 2014–2020, as-of 2024-01-03) → §3.2 preprocess → split **2020** →
**linear Cox + XGB-Cox (Optuna)** → ratings.

Paper asset: `results/reproduction/DOMAIN_02_reproduction_table.*` + `DOMAIN_02_paper_asset.md`.
See `domain2-bone-winkel-reichenbach-2024/CODE_ACCESS.md`.


#### D00 — Align loan vintage (rows + columns)


In [ ]:
run("D00_DOMAIN_02_align_loan_vintage.py")


#### D01 — Features / survival labels


In [ ]:
run("D01_DOMAIN_02_features_bondora.py")


#### D02 — Temporal split (2020 test year)


In [ ]:
run("D02_DOMAIN_02_split_temporal.py")


#### D03 — Train Cox / ratings (+ optional XGB-Cox)


In [ ]:
run("D03_DOMAIN_02_train_cox_xgb.py")


#### D03b — Attach Breslow baseline to frozen XGB-Cox (absolute S(t|x))

Required for H1 / E02–E03 on Bondora XGB. Rewrites `results/models/domain2/xgb_cox.joblib`
as `{model, features, breslow, ...}` (Booster-only backup kept). No Optuna retrain.


In [ ]:
run("D03b_DOMAIN_02_export_xgb_breslow.py")


#### D04 — Reproduction table (paper asset)

Writes mixed paper/ours/gap tables (linear + boosted Cox, Table 1 AA):
- `results/reproduction/DOMAIN_02_reproduction_table.{json,csv,md,tex}`
- `results/reproduction/DOMAIN_02_paper_asset.md` (headline paste-ready)


In [ ]:
run("D04_DOMAIN_02_gap.py")


In [ ]:
from pathlib import Path
asset = ROOT / "results/reproduction/DOMAIN_02_paper_asset.md"
table = ROOT / "results/reproduction/DOMAIN_02_reproduction_table.md"
print(asset.read_text() if asset.exists() else "missing paper asset")
print("\n--- full table ---\n")
print(table.read_text() if table.exists() else "missing table")


### Lane DOMAIN_03 — Abedi Firouzjaei (Stack Exchange / RSF)

**Fase A implemented** (D00–D03). Author `user_features` via B02; θ∈{24,36}; behavioural/content/combined; 5-fold×30 RSF.

Paper asset: `results/reproduction/DOMAIN_03_reproduction_table.*` + `DOMAIN_03_paper_asset.md`.  
See `domain3-abedi-2022/CODE_ACCESS.md`. Gaps Table 8 ~−0.03…−0.08 (sksurv vs PySurvival).


#### D00 — Features / survival labels


In [ ]:
run("D00_DOMAIN_03_features_stackexchange.py")


#### D01 — CV protocol freeze (5-fold × 30)


In [ ]:
run("D01_DOMAIN_03_cv_protocol.py")


#### D02 — Train RSF (Table 8 grid)

Uses `CDS_D3_PYTHON` / resolved `d3-pysurvival` when available (see `domain3-abedi-2022/CODE_ACCESS.md`).


In [ ]:
run("D02_DOMAIN_03_train_rsf.py")
# Freeze sksurv RSF estimators for Block E (main-trunk interpreter; not PySurvival)
run("D02b_DOMAIN_03_export_ladder_rsf.py")


#### D03 — Reproduction table (paper asset)

`results/reproduction/DOMAIN_03_reproduction_table.{json,csv,md,tex}` + `DOMAIN_03_paper_asset.md`


In [ ]:
run("D03_DOMAIN_03_gap.py")


In [ ]:
from pathlib import Path
asset = ROOT / "results/reproduction/DOMAIN_03_paper_asset.md"
print(asset.read_text() if asset.exists() else "run D03 first")


### Merge D99 *(after all lanes)*

Script ready: `D99_ALL_reproduction_report.py`.  
Merges `DOMAIN_01..03_reproduction_table.json` → `results/reproduction/report.{json,csv,md}`.

Run when closing Fase A / opening the D→E gate.


In [ ]:
run("D99_ALL_reproduction_report.py")


---
## Block E — Anchor evaluation ladder (roadmap Phase B)

Keep models from Block D **frozen**. Only the evaluation lens changes.
Script names parallel the anchor **double-helix ladder** rungs.

| ID | Script | Anchor rung |
|----|--------|-------------|
| E00 | `E00_load_frozen_models.py` | gate (hash + sanity-load) |
| E01 | `E01_rung1_discrimination.py` | 1 discrimination |
| E02 | `E02_rung2_calibration.py` | 2 calibration |
| E03 | `E03_rung3_proper_scores.py` | 3 proper scores |
| E04 | `E04_rung4_dependent_censoring.py` | 4 dependent censoring |
| E05 | `E05_rung5_competing_risks.py` | 5 competing risks |
| E06 | `E06_ladder_summary.py` | rollup (**after** F00/H04 — see Block G) |

This cell runs **E00–E05** only. `E06` runs in Block G after probes so H4 flags are fresh.

Re-run E00 after any Domain lane refresh. Metric bodies fill rung-by-rung.

**D2 XGB (2026-07-12):** after `D03b`, E03 reports `cox_xgboost` live via Breslow + sksurv IPCW-IBS
(`n_grid=40`, fresh subprocess — SurvivalEVAL must not load before the XGB Booster on this stack).
Ladder snapshot: classical IBS≈0.717 (SurvivalEVAL) / sksurv≈0.547; XGB sksurv IBS≈0.481.
H01 must rank D2 models on a **shared** IBS backend (sksurv).


In [ ]:
run("E00_load_frozen_models.py")
run("E01_rung1_discrimination.py")
run("E02_rung2_calibration.py")
run("E03_rung3_proper_scores.py")
run("E04_rung4_dependent_censoring.py")
run("E05_rung5_competing_risks.py")
# E06 after F00/H04 — see Block G
print("Block E: E00–E05 done; E06 deferred until after probes/H4")


---
## Block AN — Anchor harness check *(synthetic Lillelund)*

Parallel to Domain lanes. Table 2 = oracle truth; Figure 5 = censored-metric errors (ladder thesis).

| ID | Script | Status |
|----|--------|--------|
| AN00 | `AN00_ANCHOR_fetch_code.py` | fetch author code |
| AN01 | `AN01_ANCHOR_synthetic_dgp.py` | freeze DGP protocol |
| AN02 | `AN02_ANCHOR_run_ladder_hypo.py` | live (`--full` = 100 seeds) |
| AN03 | `AN03_ANCHOR_gap.py` | long reproduction table |
| AN04 | `AN04_ANCHOR_compare_table.py` | Table 2 compare |
| AN05 | `AN05_ANCHOR_figure5_compare.py` | Figure 5 compare |
| AN05b | `AN05b_uno_backend_divergence.py` | SurvivalEVAL vs sksurv Uno C (τ=0.50) |


In [ ]:
run("AN00_ANCHOR_fetch_code.py")
run("AN01_ANCHOR_synthetic_dgp.py")
# Paper Table 2 / Figure 5: n_seeds=100 (~10–15 min). Smoke: drop extra_args.
run("AN02_ANCHOR_run_ladder_hypo.py", extra_args=["--full"])
run("AN03_ANCHOR_gap.py")
run("AN04_ANCHOR_compare_table.py")
run("AN05_ANCHOR_figure5_compare.py")
# §5.2: SurvivalEVAL vs sksurv Uno bias under dependent censoring (τ=0.50)
run("AN05b_uno_backend_divergence.py")


---
## Block F — Failure-mode probes (roadmap Phase C)

| Script | Domain | Hypothesis |
|--------|--------|------------|
| `F00_leakage_ablation_backblaze.py` | 1 | H4 companion ablated Cox (same H6a pop) |
| `F00_sens1_leave_one_out.py` | 1 | H4 primary LOO survey |
| `F00b_h4_cluster_ablation.py` | 1 | H4 extension — age/usage & degradation clusters (`--full`) |
| `F01_competing_risks_bias_bondora.py` | 2 | H3 |
| `F02_temporal_brier_stackexchange.py` | 3 | H5 (`--full`) |
| `F03_probes_report.py` | all | rollup — **after** H01/H04 (Block G) |

This cell runs F00 → F02 (incl. F00b). **F03** and **E06** run in Block G after H01/H04 (same order as Block G below).

**H4 after H6a:** LOO max ΔC≈0.01 ≪ 0.03 → **0 floor hits** (H4 does not reject). Cluster A (SMART 9/240/241) clears the same rule post hoc (§4.6).


In [ ]:
run("F00_leakage_ablation_backblaze.py")
run("F00_sens1_leave_one_out.py")
run("F00b_h4_cluster_ablation.py", extra_args=["--full"])  # H4 cluster extension (§4.6)
run("F01_competing_risks_bias_bondora.py")
run("F02_temporal_brier_stackexchange.py", extra_args=["--full"])
# F03 + E06 after H01/H04 — see Block G
print("Block F: F00–F02 (+F00b) done; F03/E06 deferred until after H01/H04")


---
## Block G — Rigor + reusable harness (roadmap Phase D, close)

| Script | Role |
|--------|------|
| `H01_ranking_inversion.py` | H1 — τ_K + subject bootstrap (`--full` → B=1000) |
| `H04_paired_bootstrap_loo.py` | H4 — paired bootstrap LOO hits (`--full` → B=1000) |
| `F03_probes_report.py` | probes rollup (after H4) |
| `E06_ladder_summary.py` | ladder rollup (after F00/H04) |
| `G00_leakage_controls_audit.py` | leakage/rigor checklist |
| `G01_export_harness.py` | export `results/harness/` |
| `G03_holm_family.py` | **Holm outer + H_meta (formal)** |
| `G02_final_verdict_table.py` | paper verdict table — **reads G03** |

**Formal chain:** H01/H04 → F03 → E06 → G00/G01 → G03 → G02 → **Block P**.  
H1: D2 τ=−1 observed; boot p(τ=1)≈0.082 → no C00.1 reject.  
H4: no LOO hits on H6a → `p=1`, no reject.  
H_meta still **True** (**3/5**: H2, H3, H5).


In [ ]:
run("H01_ranking_inversion.py", extra_args=["--full"])
run("H04_paired_bootstrap_loo.py", extra_args=["--full"])
run("F03_probes_report.py")
run("E06_ladder_summary.py")  # after F00/H04 so H4 flags are fresh
run("G00_leakage_controls_audit.py")
run("G01_export_harness.py")
run("G03_holm_family.py")
run("G02_final_verdict_table.py")  # after G03 — authority = Holm formal
print("Block G: H→F03→E06→G00/G01→G03→G02 — then Block P")


---
## Block P — Paper *(this block **is** the manuscript source of truth)*

Nothing in the paper is hard-coded. Pipeline:

1. **P00** inventories artefacts (SHA-256, fixed order) → `results/paper/collection/`
2. **P01** extracts scalars → `numbers.json` and paper skeleton → `PAPER.md`
3. **P02** tables T01–T08 → `results/paper/tables/` (`.tex` + `.csv`)
4. **F06** CG-IPCW copula sweep (D1) → `numbers_copula_sweep_d1.json` (+ PDF)
5. **P03** figures F01–F06 → `results/paper/figures/` (**PDF only**; needs Graphviz `dot` for F01)

Assets: T01 domains · T02 anchor oracle · T03 hypotheses · T04 reproduction · T05 H1 τ · T06 D-Cal · T07 H3 strata · T08 Holm verdict · F01 workflow · F02 H1 slopegraph · F03 H2 histogram · F04 H3 point-range · F05 H5 Brier strip · F06 CG-IPCW sweep.

Cite only via P01 numbers; regenerate tables/figures with P02/P03.


In [ ]:
run("P00_collect_artifacts.py")
run("P01_paper_numbers.py")
run("P02_paper_tables.py")
run("results/paper/scripts/F06_copula_sweep_d1.py")  # before P03 so F06 PDF is included
run("P03_paper_figures.py")
print("Block P: results/paper/{collection,numbers,tables,figures} — manuscript SoT")


---
## Done

When all blocks are implemented, a full run of this notebook is the canonical reproducibility path.  
Individual scripts remain callable from the CLI for debugging a single stage.